# Food Review Classification — Full Pipeline

รันครบใน notebook เดียว: **เทรน Baseline + XLM-R → Evaluate (HF test + holdout) → Visualize → Score**

**ก่อนรัน:** เปิด notebook จากโฟลเดอร์โปรเจกต์ (ที่มี `config.py`) และติดตั้ง dependencies

```bash
pip install -r requirements.txt
```

**ข้อมูล:** `train_reduce.csv` (เทรน), `test.csv` (HF eval), `holdout.csv` (สร้างตอน train) — ดู `config.py` และ [README2.md](README2.md)

## 1. Setup

In [ ]:
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import IFrame, display

# โฟลเดอร์โปรเจกต์ (ต้องมี config.py)
ROOT = Path.cwd()
if not (ROOT / "config.py").exists():
    candidate = ROOT / "foods-review-classification"
    if (candidate / "config.py").exists():
        ROOT = candidate
    else:
        raise FileNotFoundError(
            "ไม่พบ config.py — chdir ไปที่ root ของโปรเจกต์ก่อนรัน notebook"
        )

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import config

print(f"Project root: {ROOT}")
print(f"Train data:   {config.RAW_DATA_PATH}")
print(f"Test data:    {config.WONGNAI_TEST_PATH}")
print(f"PyTorch device: {config.TORCH_DEVICE}")

## 2. ตั้งค่าการรัน (แก้ได้ตรงนี้)

In [ ]:
# ขั้นตอนที่จะรัน
RUN_TRAIN_BASELINE = True
RUN_TRAIN_XLMR = False  # เปิด True ถ้ามี GPU และต้องการเทียบ XLM-R
RUN_EVALUATE = True
RUN_EVALUATE_HOLDOUT = True  # eval บน holdout (distribution ใกล้ train)
RUN_VISUALIZE = True
RUN_SCORE = True

# ข้ามเทรนถ้ามี artifact อยู่แล้ว (เร็วขึ้นตอนรันซ้ำ)
SKIP_TRAIN_IF_ARTIFACTS_EXIST = True

# ชุดทดสอบ / output
TEST_CSV = config.WONGNAI_TEST_PATH
HOLDOUT_CSV = config.HOLDOUT_PATH
EVAL_JSON = config.DEFAULT_EVAL_REPORT
EVAL_HTML = config.DEFAULT_EVAL_VIZ
SCORED_CSV = config.DEFAULT_SCORED_OUTPUT
SCORE_MODEL = "baseline"  # "baseline" หรือ "xlmr"

## 3. ตรวจสอบข้อมูล

In [ ]:
def require_file(path: str, label: str) -> None:
    if not Path(path).is_file():
        raise FileNotFoundError(
            f"{label} ไม่พบ: {path}\n"
            "  Wongnai: python scripts/download_wongnai.py\n"
            "  Mock:    python scripts/generate_mock_data.py"
        )

require_file(config.RAW_DATA_PATH, "Train CSV")
require_file(TEST_CSV, "Test CSV")

df_train_peek = pd.read_csv(config.RAW_DATA_PATH, nrows=3)
df_test_peek = pd.read_csv(TEST_CSV, nrows=3)
print(f"Train rows (file): {sum(1 for _ in open(config.RAW_DATA_PATH, encoding='utf-8')) - 1}")
print(f"Test rows (file):  {sum(1 for _ in open(TEST_CSV, encoding='utf-8')) - 1}")
display(df_train_peek)
display(df_test_peek)

## 4. เทรน Baseline (TF-IDF + XGBoost)

In [ ]:
def baseline_artifacts_ready() -> bool:
    paths = [config.TFIDF_VECTORIZER_PATH, config.XGB_MODEL_PATH]
    meta_path = Path(config.BASELINE_META_PATH)
    if meta_path.is_file():
        import json

        meta = json.loads(meta_path.read_text(encoding="utf-8"))
        if meta.get("use_char_tfidf"):
            paths.append(config.CHAR_TFIDF_VECTORIZER_PATH)
        if meta.get("use_lsa"):
            paths.append(config.LSA_TRANSFORMER_PATH)
    return all(Path(p).is_file() for p in paths)


baseline_ready = baseline_artifacts_ready()

if RUN_TRAIN_BASELINE:
    if SKIP_TRAIN_IF_ARTIFACTS_EXIST and baseline_ready:
        print("ข้าม train_baseline — พบ artifact แล้ว")
    else:
        import train_baseline

        train_baseline.main()
else:
    print("ข้าม RUN_TRAIN_BASELINE")

## 5. เทรน XLM-R (PyTorch manual loop)

In [ ]:
xlmr_ready = (Path(config.XLMR_ARTIFACTS_DIR) / "config.json").is_file()

if RUN_TRAIN_XLMR:
    if SKIP_TRAIN_IF_ARTIFACTS_EXIST and xlmr_ready:
        print("ข้าม train_xlmr — พบ artifact แล้ว")
    else:
        import train_xlmr

        train_xlmr.main()
else:
    print("ข้าม RUN_TRAIN_XLMR")

## 6. Evaluate (เปรียบเทียบ Baseline vs XLM-R)

In [ ]:
if RUN_EVALUATE:
    from evaluate import ensure_artifacts, evaluate_model, majority_baseline_metrics, print_metrics
    from score import prepare_scoring_dataframe, predict_baseline, predict_xlmr

    model = "both" if RUN_TRAIN_XLMR else "baseline"
    ensure_artifacts(model)

    def eval_one(path: str, label: str) -> dict:
        df = prepare_scoring_dataframe(path)
        y_true = df["user_rating"].values.astype(int)
        out = {"input": path, "models": {}}
        if model in ("baseline", "both"):
            exp_b = predict_baseline(df)
            out["models"]["baseline"] = evaluate_model(
                f"Baseline — {label}", df, exp_b
            )
        if model == "both":
            exp_x = predict_xlmr(df)
            out["models"]["xlmr"] = evaluate_model(f"XLM-R — {label}", df, exp_x)
        maj = majority_baseline_metrics(y_true)
        print_metrics(
            f"Majority (4★) — {label}",
            maj,
            y_true,
            __import__("numpy").full(len(y_true), 4, dtype=int),
        )
        return out

    results = eval_one(TEST_CSV, "HF test")
    if RUN_EVALUATE_HOLDOUT and Path(HOLDOUT_CSV).is_file():
        holdout_results = eval_one(HOLDOUT_CSV, "holdout")
        results["holdout"] = holdout_results

    Path(EVAL_JSON).parent.mkdir(parents=True, exist_ok=True)
    with open(EVAL_JSON, "w", encoding="utf-8") as f:
        json.dump(results, f, indent=2, ensure_ascii=False)
    print(f"\nSaved: {EVAL_JSON}")

    if "baseline" in results["models"]:
        summary = pd.DataFrame(
            {
                "metric": ["MAE", "RMSE", "Accuracy", "F1 macro"],
                "baseline_hf_test": [
                    results["models"]["baseline"]["mae"],
                    results["models"]["baseline"]["rmse"],
                    results["models"]["baseline"]["accuracy"],
                    results["models"]["baseline"]["f1_macro"],
                ],
            }
        )
        if model == "both" and "xlmr" in results["models"]:
            summary["xlmr"] = [
                results["models"]["xlmr"]["mae"],
                results["models"]["xlmr"]["rmse"],
                results["models"]["xlmr"]["accuracy"],
                results["models"]["xlmr"]["f1_macro"],
            ]
        display(summary.style.format(precision=4))
else:
    print("ข้าม RUN_EVALUATE")

## 7. Visualize (HTML dashboard)

In [ ]:
if RUN_VISUALIZE:
    if not Path(EVAL_JSON).is_file():
        raise FileNotFoundError(f"ไม่พบ {EVAL_JSON} — รัน Evaluate ก่อน")

    from visualize_eval import load_report, render_html

    report = load_report(EVAL_JSON)
    html = render_html(report, EVAL_JSON)
    Path(EVAL_HTML).parent.mkdir(parents=True, exist_ok=True)
    with open(EVAL_HTML, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"Saved: {EVAL_HTML}")
    print("แสดงตัวอย่างด้านล่าง (หรือเปิดไฟล์ในเบราว์เซอร์)")
    rel_html = Path(EVAL_HTML).relative_to(ROOT).as_posix()
    display(IFrame(src=rel_html, width="100%", height=720))
else:
    print("ข้าม RUN_VISUALIZE")

## 8. Score (ทำนาย + anomaly บนชุด test)

In [ ]:
if RUN_SCORE:
    from score import get_hex_color, predict_baseline, predict_xlmr, prepare_scoring_dataframe

    df_score = prepare_scoring_dataframe(TEST_CSV)
    if SCORE_MODEL == "baseline":
        df_score["ai_expected_rating"] = predict_baseline(df_score)
    elif SCORE_MODEL == "xlmr":
        df_score["ai_expected_rating"] = predict_xlmr(df_score)
    else:
        raise ValueError("SCORE_MODEL must be 'baseline' or 'xlmr'")

    df_score["ai_hex_color"] = df_score["ai_expected_rating"].apply(get_hex_color)
    df_score["delta"] = np.abs(df_score["user_rating"] - df_score["ai_expected_rating"])
    df_score["is_anomaly"] = df_score["delta"] >= config.ANOMALY_THRESHOLD

    Path(SCORED_CSV).parent.mkdir(parents=True, exist_ok=True)
    df_score.to_csv(SCORED_CSV, index=False)
    print(f"Saved: {SCORED_CSV} (model={SCORE_MODEL})")
    print(f"Anomalies (delta >= {config.ANOMALY_THRESHOLD}): {df_score['is_anomaly'].sum()} / {len(df_score)}")
    display(df_score.head(10))
else:
    print("ข้าม RUN_SCORE")

## 9. สรุปผลลัพธ์

In [ ]:
outputs = {
    "Baseline artifacts": config.BASELINE_ARTIFACTS_DIR,
    "XLM-R artifacts": config.XLMR_ARTIFACTS_DIR,
    "Eval JSON": EVAL_JSON if Path(EVAL_JSON).exists() else "(ยังไม่สร้าง)",
    "Eval HTML": EVAL_HTML if Path(EVAL_HTML).exists() else "(ยังไม่สร้าง)",
    "Scored CSV": SCORED_CSV if Path(SCORED_CSV).exists() else "(ยังไม่สร้าง)",
}
for label, path in outputs.items():
    print(f"{label:20} → {path}")

print("\nPipeline เสร็จแล้ว")